# Embedding Evaluation — Metadata Notebook

End-to-end walkthrough using `emb_tight_meta.json` and `emb_sparse_meta.json`. Both files supply a `metadata` field alongside embeddings, enabling graded nDCG and per-attribute KPIs.

**Tight** — orthogonal class prototypes, tiny per-sample noise. Intra-class cosine ≈ 0.99, inter-class ≈ 0.00. All KPIs near-perfect.

**Sparse** — nearby class prototypes (~60° apart), large per-sample noise. Classes overlap heavily — gap ≈ 0.07, purity@5 ≈ 0.56.

Each image belongs to a metadata group (`corn_HB-25000SBC`, `corn_nikon_d610`, etc.) that carries `class_name` and `attributes` (`camera`, `growth_stage`). This unlocks:

* `knn_metadata_ndcg` — graded 0–3 relevance (explicit positive > same class + attrs > same class > other)
* `knn_attribute_ndcg` — one binary nDCG per attribute key (`camera`, `growth_stage`)

In [16]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.schemas.evaluate import MetadataGroup
from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

---

## Tight Scenario

### Load Data

In [17]:
payload_path = Path("emb_tight_meta.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_tight_meta.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
metadata: dict[str, MetadataGroup] = {key: MetadataGroup(**group) for key, group in payload["metadata"].items()}

print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")
print(f"Metadata groups: {list(metadata.keys())}")

Loaded 23 embeddings  (dim=32)
Metadata groups: ['corn_HB-25000SBC', 'corn_nikon_d610', 'soybean_HB-25000SBC', 'soybean_anafi']


### Run Evaluation

In [18]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
    metadata=metadata,
)

print_result(result)

Confusion matrix: 100%|██████████| 10/10 [00:00<00:00, 16.68step/s]        

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.4509  std=0.4343  (p05=-0.0204  p50=0.4125  p95=0.9916)
  centroid cosine    : mean=0.6890  std=0.1954  norm=0.6890
  intra/inter gap    : 0.7908  (intra=0.8010  inter=0.0102)
  effective_rank     : 1.76  (ratio=0.0550  dim=32)
  uniformity         : -1.0040
  alignment          : 0.2341

  hubness@5         : mean=5.0000  std=2.5195  p95=10.6000
  hubness@10        : mean=10.0000  std=4.8094  p95=18.6000
  knn_radius@5         : mean=0.8891  std=0.1559  p05=0.5552  p95=0.9873
  knn_radius@10        : mean=0.5902  std=0.4254  p05=0.0045  p95=0.9805
  mean_top_k_sim@5         : mean=0.9494  std=0.0609  p05=0.8210  p95=0.9903
  mean_top_k_sim@10        : mean=0.8034  std=0.1903  p05=0.5766  p95=0.9869
  outlier_score@5         : mean=0.0506  std=0.0609  p95=0.1790
  outlier_score@10        : m

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [19]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [20]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [21]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [22]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [23]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...


---

## Sparse Scenario

Same image paths, metadata groups, and attribute structure as the tight scenario, but class prototypes are close together (~60° apart) and per-sample noise is large. Compare KPIs and plots directly against the tight scenario above to see how embedding quality degrades.

### Load Data

In [24]:
sparse_path = Path("emb_sparse_meta.json")
with open(sparse_path) as f:
    sparse_payload = json.load(f)

sparse_embeddings: dict[str, list[float]] = sparse_payload["embeddings"]
sparse_metadata: dict[str, MetadataGroup] = {
    key: MetadataGroup(**group) for key, group in sparse_payload["metadata"].items()
}

print(f"Loaded {len(sparse_embeddings)} sparse embeddings  (dim={len(next(iter(sparse_embeddings.values())))})")
print(f"Metadata groups: {list(sparse_metadata.keys())}")

Loaded 23 sparse embeddings  (dim=32)
Metadata groups: ['corn_HB-25000SBC', 'corn_nikon_d610', 'soybean_HB-25000SBC', 'soybean_anafi']


### Run Evaluation

In [25]:
sparse_result = run_evaluation(
    image_embeddings=sparse_embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
    metadata=sparse_metadata,
)

print_result(sparse_result)

Confusion matrix: 100%|██████████| 10/10 [00:00<00:00, 32.37step/s]        

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.2989  std=0.2826  (p05=-0.1287  p50=0.3227  p95=0.7003)
  centroid cosine    : mean=0.5739  std=0.1942  norm=0.5739
  intra/inter gap    : 0.4894  (intra=0.5155  inter=0.0262)
  effective_rank     : 6.25  (ratio=0.1952  dim=32)
  uniformity         : -2.2539
  alignment          : 0.7599

  hubness@5         : mean=5.0000  std=2.5022  p95=8.0000
  hubness@10        : mean=10.0000  std=3.9009  p95=16.0000
  knn_radius@5         : mean=0.5065  std=0.1601  p05=0.2282  p95=0.6873
  knn_radius@10        : mean=0.3718  std=0.1806  p05=0.0325  p95=0.5393
  mean_top_k_sim@5         : mean=0.6078  std=0.1042  p05=0.4109  p95=0.7241
  mean_top_k_sim@10        : mean=0.5171  std=0.1358  p05=0.3094  p95=0.6599
  outlier_score@5         : mean=0.3922  std=0.1042  p95=0.5891
  outlier_score@10        : me

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [26]:
plot_knn_confusion(sparse_result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [27]:
plot_cosine_similarity(sparse_embeddings, sparse_result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [28]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [29]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [30]:
plot_lle(sparse_embeddings, sparse_result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...
